# Global Analysis of All Reconstructed Matrices

In [222]:
from pathlib import Path
import warnings
import networkx as nx
import json
import numpy as np
import pandas as pd
import torch

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


# Step 1: Load Full Reconstructed Dataset (Local PC)
Load reconstructed matrices.

In [223]:
WINDOW_LENGTH = 252
STRIDE = 5
FILE_NAME = 'data_00_20'
DATASET_NAME = f'{FILE_NAME}_w{WINDOW_LENGTH}_s{STRIDE}'
RUN_NAME = 'linearAE_614dim_0001'
DATASET = 'test'  # 'test' or 'all'
FILE_NAME = f'{DATASET}_reconstructed_{RUN_NAME}.pt'

project_root = Path.cwd().resolve().parent
dataset_dir = project_root / 'models' / DATASET_NAME / 'linearAE' / RUN_NAME / 'analysis_outputs'
ALL_RECONSTRUCTED_MATRIX_FILE = dataset_dir / f'{DATASET}_reconstructed_{RUN_NAME}.pt'
RESULTS_FOLDER = project_root / 'results' / DATASET_NAME / 'reconstruction_analysis' / RUN_NAME
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)


print(f'Dataset selected: {DATASET_NAME}')
print(f'All Reconstructed Matrix: {ALL_RECONSTRUCTED_MATRIX_FILE.name}')

Dataset selected: data_00_20_w252_s5
All Reconstructed Matrix: test_reconstructed_linearAE_614dim_0001.pt


In [224]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')
    corr_tensor = payload.get('corr_tensor', None)
    recon_tensor = payload.get('corr_tensor_reconstructed', None)
    indices = payload.get('indices', None)
    meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')
    if recon_tensor is None:
        raise KeyError('corr_tensor_reconstructed key not found in .pt file')
    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')
    if recon_tensor.ndim != 3 or recon_tensor.shape[1] != recon_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor_reconstructed: {recon_tensor.shape}')

    return corr_tensor.float(), recon_tensor.float(), indices, meta


gt_corr, recon_corr, indices, meta = load_corr_payload(ALL_RECONSTRUCTED_MATRIX_FILE)

print(f'Ground truth correlation tensor shape: {gt_corr.shape}')
print(f'Reconstructed correlation tensor shape: {recon_corr.shape}')
print(f'Indices: {indices[:10]}')
print(f'Metadata keys: {list(meta.keys())}')
print(gt_corr[:1,:5,:5])
print(recon_corr[:1,:5,:5])

num_windows, num_assets, _ = gt_corr.shape
print(f'\nNumber of windows: {num_windows}, Number of assets: {num_assets}')


Ground truth correlation tensor shape: torch.Size([133, 100, 100])
Reconstructed correlation tensor shape: torch.Size([133, 100, 100])
Indices: [847, 848, 849, 850, 851, 852, 853, 854, 855, 856]
Metadata keys: ['cholesky_L', 'indices', 'split', 'meta', 'corr_tensor_reconstructed', 'tickers']
tensor([[[1.0000, 0.2769, 0.0446, 0.1545, 0.3410],
         [0.2769, 1.0000, 0.1668, 0.3001, 0.3637],
         [0.0446, 0.1668, 1.0000, 0.2089, 0.2174],
         [0.1545, 0.3001, 0.2089, 1.0000, 0.2227],
         [0.3410, 0.3637, 0.2174, 0.2227, 1.0000]]])
tensor([[[1.0000, 0.2767, 0.1798, 0.1883, 0.2050],
         [0.2767, 1.0000, 0.2411, 0.2518, 0.3090],
         [0.1798, 0.2411, 1.0000, 0.1574, 0.1962],
         [0.1883, 0.2518, 0.1574, 1.0000, 0.2363],
         [0.2050, 0.3090, 0.1962, 0.2363, 1.0000]]])

Number of windows: 133, Number of assets: 100


# Errors Statistics (MSE,MAE,Frobenius on full Reconstructed Dataset)

In [225]:
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    diff = original - reconstructed

    mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
    mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
    fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    summary = pd.DataFrame({
        'MSE': mse_per_matrix,
        'MAE': mae_per_matrix,
        'Frobenius': fro_per_matrix,
    })

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats

In [226]:
gt_corr_np = gt_corr.cpu().numpy()
recon_corr_np = recon_corr.cpu().numpy()

errors_df, errors_stats = reconstruction_errors(gt_corr_np, recon_corr_np)

print(f'Reconstruction error statistics ({DATASET} dataset):')
display(errors_stats)
# 7. Preparazione payload JSON
file_path = RESULTS_FOLDER / 'reconstruction_errors.json'
# Inverti righe e colonne con .T
errors_stats_dict = errors_stats.T.to_dict()

Reconstruction error statistics (test dataset):


,mean,std,min,median,max
MSE,0.012727,0.002746,0.009621,0.011326,0.018187
MAE,0.087630,0.008378,0.077324,0.084362,0.103706
Frobenius,11.219542,1.183253,9.808923,10.642398,13.485748


In [227]:
def mst_edge_set(mst) -> set:
    return {frozenset(edge) for edge in mst.edges()}

def degree_distribution(mst, n_assets: int) -> np.ndarray:
    degrees = np.array([deg for _, deg in mst.degree()], dtype=int)
    counts = np.bincount(degrees, minlength=n_assets)
    return counts / counts.sum()

def compare_mst_metrics(mst_orig, mst_recon, n_assets: int, top_k: int):
    # Edges overlap and Jaccard
    edges_orig = mst_edge_set(mst_orig)
    edges_recon = mst_edge_set(mst_recon)
    common_edges = len(edges_orig & edges_recon)
    total_edges = max(n_assets - 1, 1)
    edge_overlap_pct = 100.0 * common_edges / total_edges
    edge_jaccard_pct = 100.0 * common_edges / max(len(edges_orig | edges_recon), 1)

    # Degree distribution and L1 distance
    dist_orig = degree_distribution(mst_orig, n_assets)
    dist_recon = degree_distribution(mst_recon, n_assets)
    degree_l1 = float(np.sum(np.abs(dist_orig - dist_recon)))

    # Average path length (weighted by distance)
    avg_path_len = nx.average_shortest_path_length(mst_orig, weight='weight')
    avg_path_len_recon = nx.average_shortest_path_length(mst_recon, weight='weight')
    avg_path_len_diff = float(abs(avg_path_len - avg_path_len_recon))

    # Average path length (unweighted, treating all edges as length 1)
    avg_path_len_unw = nx.average_shortest_path_length(mst_orig)
    avg_path_len_unw_recon = nx.average_shortest_path_length(mst_recon)
    avg_path_len_unw_diff = float(abs(avg_path_len_unw - avg_path_len_unw_recon))

    # Betweenness centrality top-k overlap
    top_k = int(min(top_k, n_assets))
    bet_orig = nx.betweenness_centrality(mst_orig, weight='weight', normalized=True)
    bet_recon = nx.betweenness_centrality(mst_recon, weight='weight', normalized=True)
    top_orig = {n for n, _ in sorted(bet_orig.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    top_recon = {n for n, _ in sorted(bet_recon.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    bet_overlap_pct = 100.0 * len(top_orig & top_recon) / max(top_k, 1)

    return {
        'edge_overlap_pct': edge_overlap_pct,
        'edge_jaccard_pct': edge_jaccard_pct,
        'degree_l1': degree_l1,
        'avg_path_len': float(avg_path_len),
        'avg_path_len_recon': float(avg_path_len_recon),
        'avg_path_len_diff': avg_path_len_diff,
        'avg_path_len_unweighted': float(avg_path_len_unw),
        'avg_path_len_unweighted_recon': float(avg_path_len_unw_recon),
        'avg_path_len_unweighted_diff': avg_path_len_unw_diff,
        'betweenness_topk_overlap_pct': bet_overlap_pct,
    }

In [228]:
def sanitize_correlation_matrix(corr_matrix):
    corr_matrix = np.clip(corr_matrix, -1.0, 1.0)
    corr_matrix = (corr_matrix + corr_matrix.T) / 2.0
    np.fill_diagonal(corr_matrix, 1.0)
    
    return corr_matrix

def build_distance_matrix(corr_matrix):
    dist_matrix = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - corr_matrix)))
    dist_matrix = (dist_matrix + dist_matrix.T) / 2.0
    np.fill_diagonal(dist_matrix, 0.0)
    
    return dist_matrix

def corr_to_mst(corr_matrix):
    """
    Converte una matrice di correlazione in un oggetto MST di NetworkX.
    Usa la metrica di distanza d = sqrt(2 * (1 - rho))
    """
    # 1. Calcolo della matrice delle distanze
    corr_matrix = sanitize_correlation_matrix(corr_matrix)
    dist_matrix = build_distance_matrix(corr_matrix)
    
    # 2. Creazione del grafo completo
    G = nx.from_numpy_array(dist_matrix)
    
    # 3. Calcolo del Minimum Spanning Tree
    mst = nx.minimum_spanning_tree(G, weight='weight')
    
    return mst

In [229]:
# Inizializziamo una lista per contenere i risultati di ogni coppia di matrici
all_metrics = []

n_samples = gt_corr_np.shape[0]
n_assets = gt_corr_np.shape[1]
top_k_centrality = 10

print(f"Inizio elaborazione di {n_samples} matrici...")

for i in range(n_samples):
    # 1. Estrazione della singola matrice (2D)
    curr_gt = gt_corr_np[i]
    curr_recon = recon_corr_np[i]
    
    # 2. Generazione degli MST per questa istanza
    mst_gt = corr_to_mst(curr_gt)
    mst_recon = corr_to_mst(curr_recon)
    
    # 3. Calcolo metriche
    metrics = compare_mst_metrics(
        mst_gt, 
        mst_recon, 
        n_assets=n_assets, 
        top_k=top_k_centrality
    )
    
    # Aggiungiamo alla lista
    all_metrics.append(metrics)
    
    # Feedback ogni 50 iterazioni per monitorare il progresso
    if (i + 1) % 10 == 0:
        print(f"Elaborate {i + 1}/{n_samples} matrici...")

# 4. Creazione DataFrame e calcolo della media
results_df = pd.DataFrame(all_metrics)

# 5. Calcolo delle statistiche aggregate
stats_df = results_df.describe().T 

# 6. Conversione in un dizionario strutturato
# 'index' orient crea un dizionario dove le chiavi sono le metriche
stats_dict = stats_df.to_dict(orient='index')

# 7. Salvataggio unico di tutte le metriche
combined_metrics = {
    'reconstruction_errors': errors_stats_dict,
    'mst_metrics': stats_dict,
}

with open(file_path, 'w') as f:
    json.dump(combined_metrics, f, indent=4)

print(f"Statistiche salvate con successo in: {file_path}")

# --- Visualizzazione rapida a schermo delle statistiche aggregate ---
print(f"\n--- Summary Statistics ({DATASET} dataset) ---")
display(stats_df)

Inizio elaborazione di 133 matrici...
Elaborate 10/133 matrici...
Elaborate 20/133 matrici...
Elaborate 30/133 matrici...
Elaborate 40/133 matrici...
Elaborate 50/133 matrici...
Elaborate 60/133 matrici...
Elaborate 70/133 matrici...
Elaborate 80/133 matrici...
Elaborate 90/133 matrici...
Elaborate 100/133 matrici...
Elaborate 110/133 matrici...
Elaborate 120/133 matrici...
Elaborate 130/133 matrici...
Statistiche salvate con successo in: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\results\data_00_20_w252_s5\reconstruction_analysis\linearAE_614dim_0001\reconstruction_errors.json

--- Summary Statistics (test dataset) ---


,count,mean,std,min,25%,50%,75%,max
edge_overlap_pct,133.0,22.131085,2.428380,16.161616,20.202020,22.222222,23.232323,29.292929
edge_jaccard_pct,133.0,12.463261,1.543027,8.791209,11.235955,12.500000,13.142857,17.159763
degree_l1,133.0,0.260602,0.062277,0.080000,0.220000,0.260000,0.300000,0.420000
avg_path_len,133.0,5.535370,1.527308,3.225454,3.740898,5.790275,6.701820,8.294018
avg_path_len_recon,133.0,4.974719,0.775818,3.529606,4.083016,5.205162,5.605258,6.241516
avg_path_len_diff,133.0,0.804019,0.694522,0.006847,0.273741,0.540470,1.187291,2.807786
avg_path_len_unweighted,133.0,7.130538,1.180037,5.291111,6.142222,6.835354,8.008283,10.050707
avg_path_len_unweighted_recon,133.0,5.770608,0.334553,5.073535,5.571111,5.766869,6.010303,7.182020
avg_path_len_unweighted_diff,133.0,1.448379,1.116877,0.000404,0.471515,1.179394,2.176566,4.332929
betweenness_topk_overlap_pct,133.0,27.443609,10.124538,10.000000,20.000000,20.000000,30.000000,60.000000
